# 05 — Label generation for multi-task FinBERT

This notebook produces the training labels for the three FinBERT heads described in `docs/NewsLens.md` §5.3:

| Head | Task | Label column | Source |
|---|---|---|---|
| 1 | Sentiment regression | `sentiment_label` ∈ [-1, 1] | VADER compound score on `Lsa_summary` |
| 2 | Controversy classifier | `controversy_label` ∈ {0, 1} | Std-dev of VADER across articles in same (ticker, day) cluster |
| 3 | Future movement classifier | `movement_label` ∈ {0, 1} | \|ret\| ≥ 1.5% over 3 trading days after publication |

### Subset-specific controversy definition
The FNSPID subset (`subset_news.parquet`) has no `Publisher` metadata (100% null). The spec's controversy definition — disagreement *across publishers* — cannot be applied. We substitute a weaker but workable proxy: **same-ticker same-day article clusters of size ≥ 3**, treating individual articles as the unit of disagreement. See `docs/NewsLens.md` §5.3 and the discussion in `MEMORY.md` for the justification.

### Mask columns
Not every article can be labeled for every head (e.g. an article might have no corresponding stock price 3 days ahead). Each label has a companion `*_mask` column (1 = include in loss, 0 = skip). The training loop uses these to mask per-head loss contributions cleanly.

In [2]:
import os
from pathlib import Path

import numpy as np
import polars as pl
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from tqdm.auto import tqdm

# VADER lexicon — silent download if missing
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon', quiet=True)

DATA_DIR = Path('../data')
NEWS_PATH  = DATA_DIR / 'Stock_news'  / 'subset_news.parquet'
PRICE_PATH = DATA_DIR / 'Stock_price' / 'stock_price.parquet'
OUT_DIR    = DATA_DIR / 'labels'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH   = OUT_DIR / 'multitask_labels.parquet'

## 1 — Load and normalize the news subset

We keep only the fields needed downstream, attach a stable `article_id`, and use `Lsa_summary` as the canonical text (median 547 chars — fits FinBERT's tokenizer comfortably and is pre-condensed so VADER and the model see the same input).

In [3]:
news = (
    pl.read_parquet(NEWS_PATH)
      .select([
          pl.col('Stock_symbol').alias('ticker'),
          pl.col('date_parsed').alias('pub_date'),
          pl.col('Article_title').alias('title'),
          pl.col('Lsa_summary').alias('text'),
      ])
      .drop_nulls(['ticker', 'pub_date', 'text'])
      .filter(pl.col('text').str.len_chars() > 20)
      .with_row_index('article_id')
)
print(f'articles: {news.height:,}')
print(f'tickers : {news["ticker"].n_unique()}')
print(f'date range: {news["pub_date"].min()} → {news["pub_date"].max()}')
news.head(3)

articles: 139,522
tickers : 29
date range: 2009-10-14 → 2023-12-16


article_id,ticker,pub_date,title,text
u32,str,date,str,str
0,"""GILD""",2023-08-17,"""Biotech Stock Roundup: GLTO Pl…","""Gilead Collaborates With Tenta…"
1,"""GS""",2021-01-20,"""Goldman and Morgan Stanley rai…","""UPDATE 1 - Goldman and Morgan …"
2,"""MU""",2019-12-26,"""MU February 2020 Options Begin…","""Of course, a lot of upside cou…"


## 2 — Sentiment labels (VADER)

Run VADER on each article's summary and keep the `compound` score, which is already bounded in [-1, 1]. This is weak supervision: the signal is noisy, but sufficient to anchor the sentiment regression head while the encoder learns financial-domain features. FinBERT will then sharpen these labels during fine-tuning.

VADER is pure-Python and lightweight — ~30s for 140k summaries.

In [4]:
sia = SentimentIntensityAnalyzer()
texts = news['text'].to_list()
sent = np.fromiter(
    (sia.polarity_scores(t)['compound'] for t in tqdm(texts, desc='VADER')),
    dtype=np.float32, count=len(texts),
)
news = news.with_columns(pl.Series('sentiment_label', sent))
news['sentiment_label'].describe()

VADER:   0%|          | 0/139522 [00:00<?, ?it/s]

statistic,value
str,f64
"""count""",139522.0
"""null_count""",0.0
"""mean""",0.485838
"""std""",0.504626
"""min""",-0.994
"""25%""",0.2732
"""50%""",0.6705
"""75%""",0.8807
"""max""",0.9998


## 3 — Controversy labels

**Cluster definition.** For each `(ticker, pub_date)` pair, collect all articles. Clusters of size ≥ 3 are scored by `std(sentiment_label)` within the cluster. Articles in smaller clusters are unlabeled and get `controversy_mask = 0`.

**Threshold τ.** Rather than hardcoding the paper's `τ = 0.3` (which was calibrated for multi-publisher coverage), we pick τ empirically as the 75th percentile of intra-cluster std. This targets roughly a 25% positive rate, avoiding extreme class imbalance.

In [5]:
cluster_stats = (
    news.group_by(['ticker', 'pub_date'])
        .agg([
            pl.len().alias('cluster_size'),
            pl.col('sentiment_label').std().alias('sentiment_std'),
        ])
)

# candidate clusters: size ≥ 3 with a well-defined std
valid = cluster_stats.filter(
    (pl.col('cluster_size') >= 3) & pl.col('sentiment_std').is_not_null()
)
tau = float(valid['sentiment_std'].quantile(0.75))
print(f'valid clusters (size ≥ 3): {valid.height:,}')
print(f'τ (75th pct of intra-cluster std): {tau:.4f}')

news = (
    news.join(cluster_stats, on=['ticker', 'pub_date'], how='left')
        .with_columns([
            ((pl.col('cluster_size') >= 3) & pl.col('sentiment_std').is_not_null())
                .cast(pl.Int8).alias('controversy_mask'),
            (pl.col('sentiment_std') > tau).cast(pl.Int8).alias('controversy_label'),
        ])
        # null labels → 0 (masked out anyway, but keep dtype clean)
        .with_columns(pl.col('controversy_label').fill_null(0))
)

masked = news.filter(pl.col('controversy_mask') == 1)
print(f'labelable articles        : {masked.height:,} ({masked.height/news.height:.1%})')
print(f'positive rate (among labelable): {masked["controversy_label"].mean():.1%}')

valid clusters (size ≥ 3): 18,413
τ (75th pct of intra-cluster std): 0.6068
labelable articles        : 100,086 (71.7%)
positive rate (among labelable): 22.9%


## 4 — Future movement labels

For each article, find the ticker's closing price on (or after) the publication date, then the closing price 3 trading days later. Label `1` if the absolute return is ≥ 1.5%.

**Implementation detail.** News dates don't always fall on trading days. We sort each ticker's price series, compute `close_t3 = close.shift(-3)`, then use `join_asof` with direction `forward` so an article published on a Saturday aligns with the next Monday's close.

In [6]:
prices = (
    pl.read_parquet(PRICE_PATH)
      .select([
          pl.col('ticker'),
          pl.col('date').str.to_date().alias('trade_date'),
          pl.col('close'),
      ])
      .filter(pl.col('ticker').is_in(news['ticker'].unique()))
      .sort(['ticker', 'trade_date'])
)
# close 3 trading days later, computed within each ticker
prices = prices.with_columns(
    pl.col('close').shift(-3).over('ticker').alias('close_t3')
)
print(prices.head(5))

shape: (5, 4)
┌────────┬────────────┬──────────┬──────────┐
│ ticker ┆ trade_date ┆ close    ┆ close_t3 │
│ ---    ┆ ---        ┆ ---      ┆ ---      │
│ str    ┆ date       ┆ f64      ┆ f64      │
╞════════╪════════════╪══════════╪══════════╡
│ CAT    ┆ 1970-01-02 ┆ 3.458333 ┆ 3.260417 │
│ CAT    ┆ 1970-01-05 ┆ 3.447917 ┆ 3.270833 │
│ CAT    ┆ 1970-01-06 ┆ 3.364583 ┆ 3.208333 │
│ CAT    ┆ 1970-01-07 ┆ 3.260417 ┆ 3.1875   │
│ CAT    ┆ 1970-01-08 ┆ 3.270833 ┆ 3.229167 │
└────────┴────────────┴──────────┴──────────┘


/var/folders/8m/kq6skbh91sx1ft5_jf_4gc540000gn/T/ipykernel_29615/1413281583.py:8: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col('ticker').is_in(news['ticker'].unique()))


In [7]:
# forward-asof join: align each article to the next trading day ≥ pub_date
news_sorted = news.sort(['ticker', 'pub_date'])
joined = news_sorted.join_asof(
    prices.sort(['ticker', 'trade_date']),
    left_on='pub_date', right_on='trade_date',
    by='ticker', strategy='forward',
)

joined = joined.with_columns([
    ((pl.col('close_t3') / pl.col('close')) - 1.0).alias('ret_3d'),
])
joined = joined.with_columns([
    pl.col('close').is_not_null().and_(pl.col('close_t3').is_not_null())
        .cast(pl.Int8).alias('movement_mask'),
    (pl.col('ret_3d').abs() >= 0.015).cast(pl.Int8).alias('movement_label'),
]).with_columns(
    pl.col('movement_label').fill_null(0)
)

mm = joined.filter(pl.col('movement_mask') == 1)
print(f'labelable articles       : {mm.height:,} ({mm.height/joined.height:.1%})')
print(f'positive rate (|ret|≥1.5%): {mm["movement_label"].mean():.1%}')

labelable articles       : 136,023 (97.5%)
positive rate (|ret|≥1.5%): 54.6%


/var/folders/8m/kq6skbh91sx1ft5_jf_4gc540000gn/T/ipykernel_29615/160619555.py:3: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  joined = news_sorted.join_asof(


## 5 — Train / val / test split

We split **by publication date**, not randomly. A random split would leak future information into training (the model would see tomorrow's articles during training when trying to predict tomorrow's price move). Using a temporal split forces the model to generalize forward in time — the same regime it will face in production.

- **Train:** pub_date ≤ 2021-12-31
- **Val:**   2022-01-01 ≤ pub_date ≤ 2022-12-31
- **Test:**  pub_date ≥ 2023-01-01

In [8]:
import datetime as dt

VAL_START  = dt.date(2022, 1, 1)
TEST_START = dt.date(2023, 1, 1)

joined = joined.with_columns(
    pl.when(pl.col('pub_date') < VAL_START).then(pl.lit('train'))
      .when(pl.col('pub_date') < TEST_START).then(pl.lit('val'))
      .otherwise(pl.lit('test'))
      .alias('split')
)
print(joined.group_by('split').len().sort('split'))

shape: (3, 2)
┌───────┬───────┐
│ split ┆ len   │
│ ---   ┆ ---   │
│ str   ┆ u32   │
╞═══════╪═══════╡
│ test  ┆ 28283 │
│ train ┆ 89549 │
│ val   ┆ 21690 │
└───────┴───────┘


## 6 — Final label table and save

Write a single parquet with text + all three labels + all three masks + split. This is the only file `06_finbert_multitask.ipynb` will read.

In [9]:
labels = joined.select([
    'article_id', 'ticker', 'pub_date', 'split',
    'title', 'text',
    'sentiment_label',
    'controversy_label', 'controversy_mask',
    'movement_label',    'movement_mask',
])
# sentiment is always labeled (VADER runs on every article), so no sentiment_mask

labels.write_parquet(OUT_PATH)
print(f'wrote {OUT_PATH}  ({labels.height:,} rows, {labels.estimated_size("mb"):.1f} MB)')

# summary
summary = labels.select([
    pl.col('sentiment_label').mean().alias('sent_mean'),
    pl.col('sentiment_label').std().alias('sent_std'),
    (pl.col('controversy_mask') == 1).sum().alias('n_controversy_labeled'),
    pl.col('controversy_label').filter(pl.col('controversy_mask') == 1).mean().alias('controversy_pos_rate'),
    (pl.col('movement_mask') == 1).sum().alias('n_movement_labeled'),
    pl.col('movement_label').filter(pl.col('movement_mask') == 1).mean().alias('movement_pos_rate'),
])
summary

wrote ../data/labels/multitask_labels.parquet  (139,522 rows, 86.7 MB)


sent_mean,sent_std,n_controversy_labeled,controversy_pos_rate,n_movement_labeled,movement_pos_rate
f32,f32,u32,f64,u32,f64
0.485838,0.504626,100086,0.229463,136023,0.546488


In [12]:
labels.head()

article_id,ticker,pub_date,split,title,text,sentiment_label,controversy_label,controversy_mask,movement_label,movement_mask
u32,str,date,str,str,str,f32,i8,i8,i8,i8
90071,"""CAT""",2009-11-04,"""train""","""The DJIA's Dangerous Indexing …","""While doing some research on t…",-0.9186,0,0,1,1
74505,"""CAT""",2009-11-18,"""train""","""Our Steroid Pumped Economy""","""Let me demonstrate what is pri…",0.5267,0,0,1,1
74785,"""CAT""",2010-02-05,"""train""","""Coordinated FX Intervention: H…","""You think this is good for Joh…",0.2382,0,0,1,1
87824,"""CAT""",2010-02-10,"""train""","""Warning -- Market May Cause In…","""Earnings to be reported after …",0.5994,0,0,1,1
21227,"""CAT""",2010-02-18,"""train""","""Can We Keep This Rally Going?""","""Earnings to be reported after …",0.9595,0,0,1,1
